# Bladwijzer — Berenklauw & Madeliefje detectie (YOLO11)

**Doel:** een object-detectiemodel trainen dat **berenklauw** (invasief, klasse 1) en **madeliefje** (klasse 0) herkent op foto's. Berenklauw is in Nederland invasief en gevaarlijk (brandwonden bij huidcontact + zonlicht), dus juist díe klasse missen we liever niet — dat heeft invloed op welke metriek we belangrijk vinden (zie sectie 5).

**Dataset:** ~156 geannoteerde images, gesplitst in **train/val/test (80/10/10, stratified, seed=42)**. We splitsen *op image-niveau* (alle boxes van één foto blijven in dezelfde split) zodat er geen data lekt van train naar val/test. Stratificatie zorgt dat beide klassen in elke split voorkomen — bij ~10:1 onbalans kan een random split anders een hele klasse uit val/test schoppen. De `seed=42` maakt de split reproduceerbaar.

De train-set is offline geaugmenteerd met **copy-paste voor berenklauw** (`cpaug_*` images): we plakken bestaande berenklauw-crops op andere train-images om de minderheidsklasse kunstmatig op te schalen. Dat doen we *alleen* op train, nooit op val/test — anders meet je op data die het model effectief al heeft gezien.

**Pipeline:** data inspectie → sanity check → training → validatie → test → inference demo. Deze volgorde is geen toeval: eerst snappen wat je in handen hebt (anders train je op rotte data), dan visueel checken (labels en augmentatie kunnen er stil naast zitten), dan pas trainen.

> **Belangrijk:** de **test-set** wordt pas helemaal aan het einde één keer gebruikt. Zodra je op de test-set kijkt en daarna nog beslissingen neemt (epochs, augmentatie, modelgrootte), is je test-score geen eerlijke schatting meer van hoe het model op échte nieuwe data presteert — dat heet *test-set leakage* of *overfitting op test*. Tijdens het tunen kijk je dus uitsluitend naar de val-set.

## 1. Imports & config

Alles wat we later nodig hebben (libraries, paden, seeds, klassenamen, kleuren) zetten we hier centraal bovenaan. Dat heeft drie redenen:

1. **Reproduceerbaarheid** — één plek voor `SEED` betekent dat random splits, sampling en visualisaties hetzelfde blijven tussen runs.
2. **Robuustheid** — `ROOT` wordt dynamisch bepaald zodat de notebook werkt of je hem nu vanuit `/notebooks` of vanuit de project-root start (Jupyter en PyCharm doen dat verschillend).
3. **Onderhoudbaarheid** — als de dataset verhuist hoef je maar één pad (`YOLO_DIR`) aan te passen.

`NAMES` en `COLORS` mappen we expliciet (madeliefje=geel, berenklauw=rood) zodat overlays in plots in elke sectie consistent zijn — handig om snel te zien of het model klassen verwart.

In [ ]:
from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

# Project root robuust bepalen (werkt of je nu in /notebooks of in root start)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
YOLO_DIR = ROOT / "data" / "yolo"
DATA_YAML = YOLO_DIR / "data.yaml"
RUNS_DIR = ROOT / "runs" / "detect"
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

NAMES = {0: "madeliefje", 1: "berenklauw"}
COLORS = {0: "yellow", 1: "red"}

print("ROOT     :", ROOT)
print("DATA_YAML:", DATA_YAML, "exists =", DATA_YAML.exists())

## 2. Dataset statistieken

Voordat je gaat trainen wil je nuchter weten *wat* je dataset eigenlijk is. We tellen daarom per split:

- **Aantal images** — kleine splits (vooral val/test met ~15 images) geven ruisige metrics; dat moeten we onthouden als de scores tussen epochs schommelen.
- **Boxes per klasse** — niet aantal images maar aantal *boxes* bepaalt hoeveel signaal het model per klasse krijgt. Eén foto kan tientallen madeliefjes bevatten en zo de tellingen scheef trekken.
- **Ratio madeliefje : berenklauw** — dit is dé reden voor copy-paste augmentatie en voor onze keuze van evaluatie-metrieken later. Een ratio van 10:1 betekent dat een model dat berenklauw volledig negeert nog steeds ~90% van de boxes 'goed' krijgt — dus accuracy/overall-mAP misleidt.

Door dit per split te printen check je tegelijk of de stratificatie écht gewerkt heeft (berenklauw moet in alle drie splits voorkomen).

In [ ]:
def count_split(split: str):
    img_dir = YOLO_DIR / "images" / split
    lbl_dir = YOLO_DIR / "labels" / split
    imgs = [p for p in img_dir.glob("*") if p.is_file()]
    boxes = {0: 0, 1: 0}
    for lbl in lbl_dir.glob("*.txt"):
        for line in lbl.read_text().splitlines():
            line = line.strip()
            if not line:
                continue
            cls = int(line.split()[0])
            boxes[cls] = boxes.get(cls, 0) + 1
    return len(imgs), boxes

for s in ["train", "val", "test"]:
    n, b = count_split(s)
    ratio = (b[0] / b[1]) if b[1] else float("inf")
    print(f"{s:5s}: {n:4d} images | madeliefje={b[0]:4d}, berenklauw={b[1]:4d}  (ratio {ratio:.2f}:1)")

## 3. Sanity check — visualiseer samples met bounding boxes

Statistieken vertellen niet alles: getallen kunnen kloppen terwijl de labels totaal verschoven zijn. Daarom tekenen we de bounding boxes letterlijk over de images heen. Wat we hier zoeken:

- **Kloppen de coördinaten?** YOLO gebruikt genormaliseerde `cx, cy, w, h` (0–1, center-based). Een veelgemaakte fout is verwarring met `x1, y1, x2, y2` of pixelwaarden — dat zie je meteen: boxes staan op de verkeerde plek of zijn microscopisch klein.
- **Klopt de class-id mapping?** Madeliefje moet 0 zijn (geel), berenklauw 1 (rood). Als de kleuren omgedraaid lijken, is er ergens in de conversie iets mis gegaan.
- **Zien `cpaug_*` images er natuurlijk uit?** Copy-paste plakt crops met een feathered alpha-mask, maar als de schaal of locatie raar is leert het model artefacten in plaats van berenklauw. Liever 10 minuten visueel checken dan een training van uren weggooien.
- **Per split** kijken — soms zit alle moeilijke data toevallig in val/test (of andersom), wat latere metrics verklaart.

In [ ]:
def show_samples(split: str, k: int = 6, only_prefix: str | None = None, title: str | None = None):
    img_dir = YOLO_DIR / "images" / split
    lbl_dir = YOLO_DIR / "labels" / split
    imgs = [p for p in img_dir.glob("*") if p.is_file()]
    if only_prefix:
        imgs = [p for p in imgs if p.name.startswith(only_prefix)]
    if not imgs:
        print(f"Geen images gevonden voor split={split} prefix={only_prefix}")
        return
    sel = random.sample(imgs, min(k, len(imgs)))
    cols = 3
    rows = (len(sel) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)
    for ax, p in zip(axes, sel):
        im = Image.open(p)
        W, H = im.size
        ax.imshow(im)
        ax.set_title(p.name, fontsize=8)
        ax.axis("off")
        lbl = lbl_dir / (p.stem + ".txt")
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                line = line.strip()
                if not line:
                    continue
                c, cx, cy, w, h = line.split()
                c = int(c); cx, cy, w, h = map(float, (cx, cy, w, h))
                x1, y1 = (cx - w / 2) * W, (cy - h / 2) * H
                ax.add_patch(plt.Rectangle((x1, y1), w * W, h * H,
                                           fill=False, edgecolor=COLORS[c], linewidth=2))
                ax.text(x1, max(0, y1 - 3), NAMES[c], color=COLORS[c], fontsize=8,
                        bbox=dict(facecolor="black", alpha=0.4, pad=1, edgecolor="none"))
    for ax in axes[len(sel):]:
        ax.axis("off")
    if title:
        fig.suptitle(title)
    plt.tight_layout(); plt.show()

In [ ]:
show_samples("train", k=6, title="Train samples")

In [ ]:
show_samples("train", k=6, only_prefix="cpaug_", title="Copy-paste augmented (cpaug_*)")

In [ ]:
show_samples("val", k=6, title="Val samples")
show_samples("test", k=6, title="Test samples")

## 4. Training

We trainen YOLO11 met een aantal bewuste keuzes:

- **`yolo11n.pt` (nano, pretrained)** — pretrained op COCO, dus de backbone heeft al algemene visuele features geleerd (randen, texturen, blad-achtige vormen). Met slechts ~156 images is dat goud waard: from-scratch trainen zou met deze datagrootte gegarandeerd onderpresteren. De `n`-variant is klein en snel — prima om eerst de pipeline te valideren; later kun je opschalen naar `s`/`m` als je hardware en data het toelaten.
- **`epochs=100` + `patience=30`** — 100 is een ruime bovengrens; `patience=30` doet *early stopping* zodra de val-metric 30 epochs niet verbetert. Zo voorkom je overfitting én verspil je geen tijd als het model klaar is.
- **`imgsz=640`** — YOLO-default; goed compromis tussen detail (kleine madeliefjes!) en snelheid.
- **`batch=16`** — gebruikelijk voor een nano-model op één GPU; verlaag bij OOM, verhoog als je geheugen over hebt.
- **Augmentatie sterker dan default**:
  - `mosaic=1.0` plakt 4 images samen in 1 trainbeeld — verviervoudigt effectief het aantal contexten en zorgt dat berenklauw vaker per batch verschijnt.
  - `copy_paste=0.3` (online, bovenop onze offline `cpaug_*`) plakt instances tijdens training — extra boost voor de minderheidsklasse.
  - `mixup=0.15` blendt twee images; helpt generalisatie bij kleine datasets.
  - `hsv_h/s/v` variëren kleur/verzadiging/helderheid — robuust tegen wisselende lichtomstandigheden buiten.
- **`seed=SEED`** — zelfde seed als bij de split → reproduceerbare runs, dus verschillen tussen experimenten zijn écht door je wijzigingen en niet door toeval.

> **Tip:** doe **eerst een korte run** (`epochs=20`) om te valideren dat de pipeline werkt (data laadt, loss daalt, geen NaN's). Pas daarna een volledige run van 100. Niets zo frustrerend als 4 uur trainen en dan ontdekken dat je class-ids omgedraaid stonden.

In [ ]:
model = YOLO("yolo11n.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=30,
    seed=SEED,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.3,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    project=str(RUNS_DIR),
    name="bladwijzer_v1",
    exist_ok=True,
)
run_dir = Path(results.save_dir)
print("Run dir:", run_dir)

## 5. Validatie (val-set) — per klasse

Hier meten we hoe goed het model presteert op data die het tijdens trainen nooit als gradient-signaal heeft gezien. Bij sterke klasse-onbalans is **overall mAP misleidend**: madeliefje domineert de tellingen, dus een model dat berenklauw bijna mist kan er nog 'goed' uitzien. Wat je dus écht moet lezen:

- **Per-class metrics** — vooral voor `berenklauw`:
  - **Recall (R)** = van alle echte berenklauwen, hoeveel vond het model? Hoge recall is hier prioriteit: een gemiste invasieve plant is duurder dan een vals alarm.
  - **Precision (P)** = van alle berenklauw-detecties, hoeveel klopten er? Lage precision = veel valse alarmen; vervelend maar minder erg dan missen.
  - **mAP50** = mean Average Precision bij IoU≥0.5; geeft een gebalanceerd beeld van P en R over verschillende confidence-drempels.
- **mAP50-95** is strenger (gemiddelde over IoU 0.5–0.95) en straft slordige box-localisatie af.

Op basis van deze cijfers beslis je of je nog moet **tunen** (meer data, andere augmentatie, groter model, langere training). Dit is de feedback-loop die je iteratief mag doorlopen — maar uitsluitend op val, nooit op test.

In [ ]:
val_metrics = model.val(data=str(DATA_YAML), split="val")
print(f"Overall mAP50    : {val_metrics.box.map50:.3f}")
print(f"Overall mAP50-95 : {val_metrics.box.map:.3f}")
print()
for i, name in NAMES.items():
    try:
        p = val_metrics.box.p[i]; r = val_metrics.box.r[i]; ap50 = val_metrics.box.ap50[i]
        print(f"{name:12s}  P={p:.3f}  R={r:.3f}  mAP50={ap50:.3f}")
    except IndexError:
        print(f"{name:12s}  (geen detecties)")

In [ ]:
from IPython.display import Image as IPImage
cm_path = run_dir / "confusion_matrix.png"
IPImage(filename=str(cm_path)) if cm_path.exists() else print("Geen confusion_matrix.png gevonden.")

De **confusion matrix** maakt zichtbaar *welk type* fouten het model maakt — iets dat scalar metrics als mAP nooit kunnen vertellen. Lees hem zo:

- **Diagonaal sterk** = klassen worden correct herkend.
- **Berenklauw → madeliefje** (of omgekeerd) = klasse-verwarring; mogelijk te weinig discriminatieve features → meer/diversere data of groter model helpen.
- **Berenklauw → background** = het model 'mist' berenklauw (false negatives); recall-probleem → meer berenklauw-samples, sterkere augmentatie of lagere confidence-drempel.
- **Background → berenklauw** = vals alarm (false positives); precision-probleem → vaak op te lossen met meer hard-negative (achtergrond) images.

## 6. Test-set — eindcijfer ⚠️

Dit is de **eerlijke eindmeting**. De test-set is tijdens alle voorgaande beslissingen (augmentatie, epochs, modelgrootte, drempels) ongezien gebleven — dáárom is de score hier een betrouwbare schatting van hoe het model op compleet nieuwe foto's presteert.

**Niet meer tunen na deze cel.** Zodra je op test kijkt en daarna nog wijzigt, vervuil je je eindcijfer. Als de test-score teleurstelt: noteer het eerlijk en plan een volgende iteratie met nieuwe data of een nieuwe split — niet met een nog-net-iets-anders getunede versie van dit model.

Verwacht dat de test-score iets *lager* is dan val (val-set is impliciet gebruikt om early-stopping en keuzes te sturen). Een groot gat (val veel hoger dan test) wijst op overfitting of een ongelukkige split.

In [ ]:
test_metrics = model.val(data=str(DATA_YAML), split="test")
print(f"TEST mAP50    : {test_metrics.box.map50:.3f}")
print(f"TEST mAP50-95 : {test_metrics.box.map:.3f}")
print()
for i, name in NAMES.items():
    try:
        p = test_metrics.box.p[i]; r = test_metrics.box.r[i]; ap50 = test_metrics.box.ap50[i]
        print(f"{name:12s}  P={p:.3f}  R={r:.3f}  mAP50={ap50:.3f}")
    except IndexError:
        print(f"{name:12s}  (geen detecties)")

## 7. Beste model bewaren als eindproduct

YOLO schrijft per run een `best.pt` (gewichten van de epoch met de hoogste val-mAP) en een `last.pt` (laatste epoch). Wij willen `best.pt` als ons inzetbare model.

Waarom kopiëren naar `models/bladwijzer_best.pt`?

- **Stabiel pad** — `runs/detect/bladwijzer_v1/weights/best.pt` verandert naam per experiment; `models/bladwijzer_best.pt` blijft hetzelfde, dus inference-scripts en evt. een demo-UI weten altijd waar het 'productie'-model staat.
- **Scheiding R&D vs deliverable** — `runs/` is rommelig (logs, plots, checkpoints van mislukte experimenten); `models/` bevat alleen wat je wil delen of deployen.
- **Versioneerbaar** — je kunt later `bladwijzer_best_v2.pt` toevoegen zonder de vorige te overschrijven.

In [ ]:
import shutil
best_src = run_dir / "weights" / "best.pt"
best_dst = MODELS_DIR / "bladwijzer_best.pt"
if best_src.exists():
    shutil.copy2(best_src, best_dst)
    print("Bewaard:", best_dst)
else:
    print("best.pt nog niet aanwezig — heb je al getraind?")

## 8. Inference demo — model loslaten op test-images

Metrics zijn één ding, maar een visuele demo overtuigt: hier zien we letterlijk wat het model voorspelt op losse images. We gebruiken een paar random test-images zodat je predictions kunt vergelijken met de echte annotaties uit sectie 3.

- **`conf=0.25`** — confidence-drempel. Lager (bv. 0.1) levert meer detecties maar ook meer false positives; hoger (bv. 0.5) is conservatief. Voor berenklauw-detectie (recall belangrijk) kun je later experimenteren met een lágere drempel.
- **`best.predict(...)`** geeft `Results`-objecten terug; `r.plot()` rendert de boxes + class-labels in BGR (vandaar `[..., ::-1]` voor matplotlib's RGB-verwachting).
- Dit is ook meteen de blauwdruk voor een productie-script (`src/predict.py`) of een Streamlit/Gradio-demo: zelfde 3 regels code.

In [ ]:
best = YOLO(str(best_dst)) if best_dst.exists() else model
test_imgs = list((YOLO_DIR / "images" / "test").glob("*"))
sample_imgs = random.sample(test_imgs, min(6, len(test_imgs)))
preds = best.predict(source=[str(p) for p in sample_imgs], conf=0.25, verbose=False)

cols = 3
rows = (len(preds) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes = np.array(axes).reshape(-1)
for ax, r in zip(axes, preds):
    ax.imshow(r.plot()[..., ::-1])
    ax.set_title(Path(r.path).name, fontsize=8)
    ax.axis("off")
for ax in axes[len(preds):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 9. Conclusie & vervolgstappen

Vul in na de eerste volledige run:

- **Metrics:** mAP50 / recall per klasse (vooral berenklauw).
- **Waar gaat het mis?** Lees de confusion matrix — wordt berenklauw verward met madeliefje of met background?
- **Mogelijke verbeteringen:**
  - meer berenklauw-data verzamelen (iNaturalist, GBIF, eigen foto's),
  - grotere variant (`yolo11s.pt` / `yolo11m.pt`),
  - meer epochs / langere patience,
  - hogere `copy_paste` / extra offline augmentatie runs,
  - hard-negative mining (achtergrond-images zonder annotaties).

**Eindproduct laag (buiten deze notebook):**
1. `models/bladwijzer_best.pt` (hierboven bewaard).
2. `src/predict.py` — kleine CLI: `python src/predict.py pad/naar/foto.jpg`.
3. README update met install + usage + behaalde metrics.
4. (Optioneel) Streamlit/Gradio demo-UI voor uploads.
5. (Optioneel) export naar ONNX/TFLite: `best.export(format="onnx")`.